In [ ]:
import polars as pl
import hvplot.polars
import panel as pn

# Import your custom FOF8 modules
from fof8_core.loader import FOF8Loader
from fof8_core.targets import (
    get_career_outcomes, 
    get_merit_cap_share, 
    get_peak_overall, 
    get_career_value_metrics,
    get_awards,
)

# Initialize Panel extension for rendering hvplot layouts
pn.extension()

In [ ]:
# ---------------------------------------------------------
# 1. Load and Aggregate All Targets & Ground Truth Data
# ---------------------------------------------------------
# Adjust base_path to point to your data directory
loader = FOF8Loader(base_path="../fof8-gen/data/raw", league_name="DRAFT004")
final_year = loader.final_sim_year

print("Calculating Career Outcomes...")
df_outcomes = get_career_outcomes(loader)

print("Calculating Financial Merit (Target 1)...")
df_merit = get_merit_cap_share(loader)

print("Calculating Peak Overall...")
df_peak = get_peak_overall(loader, k=3)




In [ ]:
print("Calculating Career VORP and Career Approximate Value(Target 2)...")
df_values = get_career_value_metrics(loader)

In [ ]:
print("Fetching Awards...")
df_awards = get_awards(loader, target_name="Total_Awards")

# We also need Position_Group for grouping, which isn't in career_outcomes directly.
# Let's pull the first time they appear in rookies.csv to get their true position group.
df_positions = (
    loader.scan_file("rookies.csv")
    .select(["Player_ID", "Position_Group"])
    .unique(subset=["Player_ID"]) # Ensure one row per player
    .collect()
)

In [ ]:
# ---------------------------------------------------------
# 2. Build the Master Analysis DataFrame
# ---------------------------------------------------------
print("Merging Datasets...")
df_analysis = (
    df_outcomes
    .join(df_positions, on="Player_ID", how="inner")
    .join(df_merit, on="Player_ID", how="left")
    .join(df_vorp, on="Player_ID", how="left")
    .join(df_peak, on="Player_ID", how="left")
    .join(df_awards, on="Player_ID", how="left")
    .with_columns([
        pl.col("Total_Awards").fill_null(0),
        pl.col("Career_VORP").fill_null(0.0),
        pl.col("Career_Merit_Cap_Share").fill_null(0.0),
        pl.col("Peak_Overall").fill_null(0.0)
    ])
    .with_columns([
        # Recreate the Stage 1 Sieve logic
        (pl.col("Career_Merit_Cap_Share") > 0).alias("Cleared_Sieve"),
        pl.when(pl.col("Career_Merit_Cap_Share") > 0)
          .then(pl.lit("Hit"))
          .otherwise(pl.lit("Bust"))
          .alias("Sieve_Status"),
        # Recreate the Stage 2 Expected DPO logic
        (pl.col("Peak_Overall") * pl.col("Career_Merit_Cap_Share")).alias("Actual_DPO"),
        # Bin Draft Rounds for easier plotting
        pl.when(pl.col("Draft_Round") == 1).then(pl.lit("1st Round"))
          .when(pl.col("Draft_Round").is_between(2, 3)).then(pl.lit("Day 2 (2-3)"))
          .when(pl.col("Draft_Round").is_between(4, 7)).then(pl.lit("Day 3 (4-7)"))
          .otherwise(pl.lit("UDFA"))
          .alias("Draft_Bucket")
    ])
)

# Filter out players with zero games played to clean up the plots
df_active = df_analysis.filter(pl.col("Career_Games_Played") > 0)

print(f"Dataset ready. Analyzing {len(df_active)} players who saw the field.")



In [ ]:
# ---------------------------------------------------------
# 3. Visualizations (Run these in separate cells if desired)
# ---------------------------------------------------------

# Plot 1: The Lie Detector (Financial Merit vs. On-Field VORP)
# If your target is good, this should be a tight, upward-sloping cluster.
# Watch out for players with High VORP but Negative Merit (the rookie contract trap).
plot_1 = df_active.hvplot.scatter(
    x="Career_VORP", 
    y="Career_Merit_Cap_Share", 
    by="Draft_Bucket",
    hover_cols=["Player_ID", "Position_Group", "Total_Awards", "Peak_Overall"],
    alpha=0.6,
    title="Financial Merit vs. On-Field Value (VORP)",
    xlabel="Career VORP (Starts * Positional Premium)",
    ylabel="Career Merit Cap Share (Earnings - Rookie Expectations)",
    width=800, height=500
)

# Plot 2: Distribution of Peak Overall
plot_2 = df_active.hvplot.violin(
    y="Peak_Overall", 
    by="Sieve_Status",  # Updated from Cleared_Sieve
    c="Sieve_Status",   # Updated from Cleared_Sieve
    cmap=["#ef4444", "#22c55e"], 
    title="Peak Overall Rating Distribution: Busts vs. Hits",
    ylabel="Scouted Peak Overall",
    width=600, height=500
)

# Plot 3: Award winners
plot_3 = (
    df_active.group_by("Sieve_Status") # Updated from Cleared_Sieve
    .agg(pl.col("Total_Awards").mean().alias("Avg_Awards_Won"))
    .sort("Sieve_Status")
).hvplot.bar(
    x="Sieve_Status", # Updated from Cleared_Sieve
    y="Avg_Awards_Won", 
    color="Sieve_Status", # Updated from Cleared_Sieve
    cmap=["#ef4444", "#22c55e"],
    title="Average Career Awards by Sieve Outcome",
    width=600, height=400
)

# Plot 4: The Positional Bias Check (The "Punter" test)
# What percentage of players clear the sieve, grouped by position?
df_pos_rates = (
    df_active.group_by("Position_Group")
    .agg([
        (pl.col("Cleared_Sieve").cast(pl.Int32).sum() / pl.len() * 100).alias("Hit_Rate_Pct"),
        pl.len().alias("Sample_Size")
    ])
    .filter(pl.col("Sample_Size") > 50) # Filter out tiny sample sizes
    .sort("Hit_Rate_Pct", descending=True)
)

plot_4 = df_pos_rates.hvplot.bar(
    x="Position_Group", 
    y="Hit_Rate_Pct",
    hover_cols=["Sample_Size"],
    title="% of Players Clearing the Sieve by Position",
    ylabel="Hit Rate (%)",
    color="steelblue",
    width=800, height=400
)

# Display all plots using Panel
layout = pn.Column(
    pn.Row(plot_1),
    pn.Row(plot_2, plot_3),
    pn.Row(plot_4)
)
layout

In [ ]:
player_id = 40162
# 1. Get Player Name
player_info = (
    loader.scan_file("player_information.csv")
    .filter(pl.col("Player_ID") == player_id)
    .select(["Last_Name", "First_Name"])
    .collect()
)
if not player_info.is_empty():
    name = f"{player_info[0, 'First_Name']} {player_info[0, 'Last_Name']}"
    print(f"Player: {name}")
# 2. Get Career History
# Mapping Team ID to Home_City from the file you found
history = (
    loader.scan_file("player_season_*.csv")
    .filter(pl.col("Player_ID") == player_id)
    .join(
        loader.scan_file("team_information.csv").select(["Team", "Home_City"]),
        left_on="Team", 
        right_on="Team", 
        how="left"
    )
    .select(["Year", "Home_City", "Games_Started"])
    .unique() # Since team_info has one row per year, we unique it
    .sort("Year")
    .collect()
)
print("\nCareer History:")
print(history)

In [ ]:
# 1. The Shape Test (Distribution)
# VORP should look like a bell curve centered near 0, with a heavy right tail (superstars)
# and a hard cutoff on the left (players bad enough to get benched/cut).
plot_shape = df_active.hvplot.hist(
    y="Career_VORP", 
    bins=50, 
    title="1. Distribution of Career VORP",
    xlabel="Career VORP",
    ylabel="Frequency",
    color="steelblue",
    width=600, height=400
)

# 2. The Ground Truth Proxy (HOF / Awards Test)
# Does VORP actually identify the players the simulation considers "Great"?
# Hall of Famers should have significantly higher VORP than non-HOFers.
plot_hof = df_active.hvplot.violin(
    y="Career_VORP", 
    by="Hall_of_Fame_Flag",
    c="Hall_of_Fame_Flag",
    cmap=["#94a3b8", "#eab308"], # Gray for normal, Gold for HOF
    title="2. VORP Validation: Hall of Fame Status",
    ylabel="Career VORP",
    xlabel="Is Hall of Famer? (0 = No, 1 = Yes)",
    width=600, height=400
)

# 3. Positional Equity Check
# Are we accidentally inflating specific positions? QBs will naturally have higher peaks, 
# but the medians across standard starting positions should be roughly comparable.
# If Centers have a max VORP of 5 and WRs have a max of 50, the AV weights need tweaking.
plot_equity = df_active.hvplot.box(
    y="Career_VORP", 
    by="Position_Group", 
    title="3. Positional Equity: VORP Spread by Position",
    ylabel="Career VORP",
    xlabel="Position",
    rot=45,
    width=800, height=400
)

# 4. The Draft Capital Baseline
# Even with salary completely removed from the math, 1st Round picks should naturally 
# generate more VORP simply because they are given more starting opportunities and 
# generally possess higher talent.
plot_draft = df_active.hvplot.box(
    y="Career_VORP", 
    by="Draft_Bucket", 
    title="4. Pure Talent Validation: VORP by Draft Round",
    ylabel="Career VORP",
    xlabel="Draft Capital",
    width=600, height=400
)

# Display the validation dashboard
layout_validation = pn.Column(
    pn.Row(plot_shape, plot_hof),
    pn.Row(plot_equity),
    pn.Row(plot_draft)
)
layout_validation